In [1]:
import sys
sys.path.append("..")

from sklearn.model_selection import GroupShuffleSplit
from src.data import load_cmapss

df = load_cmapss("../data/train_FD001.txt")
df["RUL"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]

constantes = ["setting_3", "s1", "s5", "s10", "s16", "s18", "s19"]
features = ["setting_1", "setting_2", "setting_3"] + [f"s{i}" for i in range(1, 22)]
utiles = [c for c in features if c not in constantes]

X = df[utiles]
y = df["RUL"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=df["unit"]))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

In [2]:
unidades_train = set(df.iloc[train_idx]["unit"])
unidades_val = set(df.iloc[val_idx]["unit"])

print(len(unidades_train), len(unidades_val))   # 80 20
print(unidades_train & unidades_val)            # set() -> vacío
print(X_train.shape, X_val.shape)

80 20
set()
(16561, 17) (4070, 17)


In [3]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def evaluar(modelo, nombre):
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    mae = mean_absolute_error(y_val, pred)
    print(f"{nombre}: RMSE={rmse:.1f}  MAE={mae:.1f}")

evaluar(DummyRegressor(strategy="mean"), "Baseline (media)")
evaluar(LinearRegression(), "Regresión lineal")

Baseline (media): RMSE=65.7  MAE=55.4
Regresión lineal: RMSE=38.1  MAE=30.1


In [4]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

evaluar(RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1), "Random Forest")
evaluar(XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42), "XGBoost")

Random Forest: RMSE=35.3  MAE=25.8
XGBoost: RMSE=35.3  MAE=25.9


In [5]:
def evaluar(modelo, nombre, y_tr, y_va):
    modelo.fit(X_train, y_tr)
    pred = modelo.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_va, pred))
    mae = mean_absolute_error(y_va, pred)
    print(f"{nombre}: RMSE={rmse:.1f}  MAE={mae:.1f}")

y_train_c = y_train.clip(upper=125)
y_val_c = y_val.clip(upper=125)

evaluar(DummyRegressor(strategy="mean"), "Baseline (media)", y_train_c, y_val_c)
evaluar(LinearRegression(), "Regresión lineal", y_train_c, y_val_c)
evaluar(RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1), "Random Forest", y_train_c, y_val_c)
evaluar(XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42), "XGBoost", y_train_c, y_val_c)

Baseline (media): RMSE=41.7  MAE=36.9
Regresión lineal: RMSE=20.4  MAE=16.5
Random Forest: RMSE=17.0  MAE=12.3
XGBoost: RMSE=16.9  MAE=12.3


In [8]:
import pandas as pd

lin = LinearRegression().fit(X_train, y_train_c)
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1).fit(X_train, y_train_c)
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42).fit(X_train, y_train_c)

test = load_cmapss("../data/test_FD001.txt")
ultimo = test.loc[test.groupby("unit")["cycle"].idxmax()]
rul_real_raw = pd.read_csv("../data/RUL_FD001.txt", header=None)[0]

for nombre, m in [("Regresión lineal", lin), ("Random Forest", rf), ("XGBoost", xgb)]:
    pred = m.predict(ultimo[utiles])
    rmse = np.sqrt(mean_squared_error(rul_real_raw, pred))
    mae = mean_absolute_error(rul_real_raw, pred)
    print(f"{nombre} (test oficial, RUL real): RMSE={rmse:.1f}  MAE={mae:.1f}")

Regresión lineal (test oficial, RUL real): RMSE=21.9  MAE=17.6
Random Forest (test oficial, RUL real): RMSE=18.1  MAE=13.0
XGBoost (test oficial, RUL real): RMSE=18.5  MAE=13.1


In [7]:
print(ultimo.shape, rul_real.shape)   # (100, 26) (100,)

(100, 26) (100,)
